# HE_Prop (Master/Prop EA) — Component Testing

This notebook breaks down **HE_Prop.mq5** (the Master/Prop EA) into testable components.
Each section maps to a logical module in the MQ5 source so you can validate behavior
before deploying to a MetaTrader 5 terminal via the Hedge Edge App Connector.

**Source:** `agents/mt5/HedgEdge property/Developer tools/HE_Prop.mq5` (v3.10)

## Architecture Overview
```
┌──────────────────────────────────────────────────────────┐
│                    HE_Prop (Master EA)                   │
│                                                          │
│  OnTradeTransaction() ─> PublishEvent()     ZMQ PUB      │
│       (instant)         "EVENT|{json}"     :51810 ──────────> subscribers
│                                                          │
│  OnTick()             ─> PublishSnapshot()  ZMQ PUB      │
│       (periodic)        "SNAPSHOT|{json}"  :51810 ──────────> subscribers
│                                                          │
│  OnTimer()            ─> PublishHeartbeat() ZMQ PUB      │
│       (5s interval)     "EVENT|{hb}"       :51810 ──────────> subscribers
│                                                          │
│  ProcessCommands()    ─> ZMQ REP           :51811 <─────────> Electron App
│       (app control)                                      │
└──────────────────────────────────────────────────────────┘
```

---
## Component 1 — Input Parameters & Configuration

The Master EA has fewer inputs than the Slave — no trade copy settings.
It publishes events; subscribers decide what to do with them.

In [ ]:
# === Component 1: Input Parameters (mirrors MQ5 `input` declarations) ===

master_config = {
    # --- License Settings ---
    "InpLicenseKey": "",
    "InpDeviceId": "",
    "InpEndpointUrl": "https://hedge-edge-app-backend-production.up.railway.app/v1/license/validate",
    "InpPollIntervalSeconds": 300,
    "InpDevMode": True,                         # << DEV MODE ON for testing

    # --- ZMQ Settings ---
    "InpDataPort": 51810,                       # PUB port for data/events
    "InpCommandPort": 51811,                    # REP port for app commands
    "InpEnableCommands": True,
    "InpEnableCurve": False,

    # --- Publish Settings ---
    "InpPublishIntervalMs": 500,                # Snapshot every 500ms
    "InpHeartbeatIntervalSec": 5,               # Heartbeat every 5s
}

print("Master config loaded:")
for k, v in master_config.items():
    print(f"  {k}: {v}")

---
## Component 2 — Shared License Key (Common Files)

Identical to HE_Hedge — reads from `Common Files/HedgeEdge/license.key`.

**MQ5 Function:** `ReadSharedLicenseKey()`

In [ ]:
import os

def find_mt5_common_files():
    appdata = os.environ.get("APPDATA", "")
    common_path = os.path.join(appdata, "MetaQuotes", "Terminal", "Common", "Files")
    return common_path if os.path.isdir(common_path) else None

def read_shared_license_key():
    common = find_mt5_common_files()
    if not common:
        print("MT5 Common Files not found")
        return ""
    key_file = os.path.join(common, "HedgeEdge", "license.key")
    if not os.path.isfile(key_file):
        print(f"No shared license file at: {key_file}")
        return ""
    with open(key_file, "r", encoding="utf-8") as f:
        key = f.read().strip()
    if len(key) < 8:
        return ""
    masked = key[:4] + "****" + key[-4:]
    print(f"Shared license key: {masked} ({len(key)} chars)")
    return key

print(f"Common Files: {find_mt5_common_files()}")
shared_key = read_shared_license_key()

---
## Component 3 — License Validation (WebRequest)

Same as the Slave — validates against the Railway backend.

**MQ5 Function:** `ValidateLicenseViaWebRequest()`

In [ ]:
import requests

def validate_license(license_key: str, account_id: str = "12345",
                     broker: str = "TestBroker", device_id: str = "dev-notebook"):
    url = master_config["InpEndpointUrl"]
    payload = {
        "licenseKey": license_key,
        "accountId": account_id,
        "broker": broker,
        "deviceId": device_id,
    }
    masked = license_key[:4] + "****" + license_key[-4:] if len(license_key) > 8 else "***"
    print(f"Validating key={masked} at {url}")
    try:
        resp = requests.post(url, json=payload, timeout=10)
        print(f"HTTP {resp.status_code}: {resp.json()}")
        return resp.json().get("valid", False)
    except Exception as e:
        print(f"Failed: {e}")
        return False

if master_config["InpDevMode"]:
    print("DEV MODE: License validation skipped")
else:
    key = shared_key or master_config["InpLicenseKey"]
    if key:
        validate_license(key)
    else:
        print("No license key available")

---
## Component 4 — ZMQ Publisher Setup

The Master binds a PUB socket on `tcp://*:51810`. All slave EAs subscribe to it.
A REP socket on `tcp://*:51811` handles app commands.

**MQ5 Function:** `InitializeZMQ()` (publisher side)

> **Prereq:** `pip install pyzmq`

In [ ]:
import zmq
import json as json_mod

def create_publisher(data_port: int):
    """Mirror of MQ5 InitializeZMQ() — PUB socket binding."""
    ctx = zmq.Context()
    pub = ctx.socket(zmq.PUB)
    
    pub.setsockopt(zmq.LINGER, 100)
    pub.setsockopt(zmq.SNDHWM, 1000)
    pub.setsockopt(zmq.SNDTIMEO, 100)
    
    endpoint = f"tcp://*:{data_port}"
    pub.bind(endpoint)
    print(f"PUB socket bound to {endpoint}")
    return ctx, pub

def create_replier(cmd_port: int):
    """Mirror of MQ5 InitializeZMQ() — REP socket for commands."""
    ctx = zmq.Context()
    rep = ctx.socket(zmq.REP)
    rep.setsockopt(zmq.LINGER, 100)
    rep.setsockopt(zmq.RCVTIMEO, 10)
    rep.setsockopt(zmq.SNDTIMEO, 1000)
    
    endpoint = f"tcp://*:{cmd_port}"
    rep.bind(endpoint)
    print(f"REP socket bound to {endpoint}")
    return ctx, rep

# --- Test ---
data_port = master_config["InpDataPort"]
cmd_port = master_config["InpCommandPort"]
print(f"Data port: {data_port}, Command port: {cmd_port}")
print("Uncomment to bind (only when MT5 EA is NOT running):")
# pub_ctx, pub_sock = create_publisher(data_port)
# rep_ctx, rep_sock = create_replier(cmd_port)

---
## Component 5 — Position Tracking (Diff Engine)

The Master gathers all open positions each tick/timer and compares with the
previous state to detect SL/TP modifications (which don't trigger `OnTradeTransaction`).

**MQ5 Struct:** `PositionInfo g_positions[]`, `g_prevPositions[]`

**MQ5 Functions:** `GatherPositions()`, `DetectSLTPChanges()`

In [ ]:
# === Component 5: Position Tracking & Diff ===
from dataclasses import dataclass
from copy import deepcopy

@dataclass
class PositionInfo:
    ticket: int
    symbol: str
    volume: float
    pos_type: str       # "BUY" or "SELL"
    entry_price: float
    current_price: float
    stop_loss: float
    take_profit: float
    profit: float
    swap: float
    commission: float
    open_time: str
    comment: str

def detect_sltp_changes(current: list[PositionInfo], previous: list[PositionInfo]) -> list[dict]:
    """Mirror of DetectSLTPChanges() — finds SL/TP modifications."""
    changes = []
    prev_map = {p.ticket: p for p in previous}
    
    for pos in current:
        prev = prev_map.get(pos.ticket)
        if not prev:
            continue
        sl_changed = abs(pos.stop_loss - prev.stop_loss) > 0.000001
        tp_changed = abs(pos.take_profit - prev.take_profit) > 0.000001
        if sl_changed or tp_changed:
            changes.append({
                "ticket": pos.ticket,
                "symbol": pos.symbol,
                "type": pos.pos_type,
                "stopLoss": pos.stop_loss,
                "takeProfit": pos.take_profit,
                "prevStopLoss": prev.stop_loss,
                "prevTakeProfit": prev.take_profit,
            })
    return changes

# --- Test ---
prev_positions = [
    PositionInfo(1001, "EURUSD", 0.10, "BUY", 1.0950, 1.0960, 1.0900, 1.1000, 10.0, 0, 0, "2025.01.01", ""),
    PositionInfo(1002, "GBPUSD", 0.05, "SELL", 1.2700, 1.2690, 1.2750, 1.2650, 5.0, 0, 0, "2025.01.01", ""),
]

curr_positions = deepcopy(prev_positions)
curr_positions[0].stop_loss = 1.0920   # SL moved up
curr_positions[0].take_profit = 1.1050  # TP moved up
# position 1002 unchanged

changes = detect_sltp_changes(curr_positions, prev_positions)
print("=== SL/TP Change Detection ===")
for c in changes:
    print(f"  #{c['ticket']} {c['symbol']}:")
    print(f"    SL: {c['prevStopLoss']:.5f} -> {c['stopLoss']:.5f}")
    print(f"    TP: {c['prevTakeProfit']:.5f} -> {c['takeProfit']:.5f}")
print(f"Total changes: {len(changes)}")

---
## Component 6 — Event Publishing

The Master publishes discrete events via ZMQ PUB with topic prefix `EVENT|`.
Each event has a monotonically increasing `eventIndex` for ordering.

Event types published:
- `POSITION_OPENED` — from `OnTradeTransaction` (DEAL_ENTRY_IN)
- `POSITION_CLOSED` — from `OnTradeTransaction` (DEAL_ENTRY_OUT)
- `POSITION_REVERSED` — from `OnTradeTransaction` (DEAL_ENTRY_INOUT)
- `POSITION_MODIFIED` — SL/TP diff in `DetectSLTPChanges()`
- `HEARTBEAT` — periodic keepalive
- `CONNECTED` / `DISCONNECTED` — lifecycle
- `ACCOUNT_UPDATE` — after each trade

**MQ5 Functions:** `PublishEvent()`, `OnTradeTransaction()`

In [ ]:
# === Component 6: Event Publishing ===
from datetime import datetime

class EventPublisher:
    """Mirror of the Master's event publishing subsystem."""
    
    def __init__(self, account_id: str = "12345"):
        self.event_index = 0
        self.account_id = account_id
        self.published = []  # log for testing
    
    def publish_event(self, event_type: str, data: dict) -> str:
        """Mirror of MQ5 PublishEvent()."""
        self.event_index += 1
        msg = {
            "type": event_type,
            "eventIndex": self.event_index,
            "timestamp": datetime.now().strftime("%Y.%m.%d %H:%M:%S"),
            "platform": "MT5",
            "accountId": self.account_id,
            "role": "master",
            "data": data,
        }
        json_str = json_mod.dumps(msg)
        topic_msg = f"EVENT|{json_str}"
        self.published.append((event_type, msg))
        return topic_msg
    
    def publish_heartbeat(self, balance=10000, equity=10050, profit=50,
                          margin=100, free_margin=9950, position_count=2) -> str:
        """Mirror of MQ5 PublishHeartbeat()."""
        data = {
            "balance": balance,
            "equity": equity,
            "profit": profit,
            "margin": margin,
            "freeMargin": free_margin,
            "positionCount": position_count,
            "isLicenseValid": True,
            "isPaused": False,
            "serverTime": datetime.now().strftime("%Y.%m.%d %H:%M:%S"),
            "serverTimeUnix": int(datetime.now().timestamp()),
        }
        return self.publish_event("HEARTBEAT", data)

# --- Test ---
pub = EventPublisher("12345")

# Simulate trade events
msg1 = pub.publish_event("POSITION_OPENED", {
    "deal": 99001, "position": 88001, "symbol": "EURUSD",
    "volume": 0.10, "price": 1.0950, "type": "BUY",
    "stopLoss": 1.0900, "takeProfit": 1.1000, "entry": "IN",
})
msg2 = pub.publish_heartbeat()
msg3 = pub.publish_event("POSITION_CLOSED", {
    "deal": 99002, "position": 88001, "symbol": "EURUSD",
    "volume": 0.10, "price": 1.1010, "profit": 60.0,
    "type": "SELL", "entry": "OUT",
})

print("=== Published Events ===")
for event_type, msg in pub.published:
    print(f"  [{msg['eventIndex']}] {event_type}")

print(f"\nSample wire format (first 200 chars):")
print(f"  {msg1[:200]}...")

---
## Component 7 — Snapshot Publishing

Every `InpPublishIntervalMs` (500ms), a full SNAPSHOT is published with topic `SNAPSHOT|`.
This serves as a reconciliation backup.

**MQ5 Functions:** `PublishSnapshot()`, `BuildFullSnapshotJson()`, `BuildPositionsJson()`

In [ ]:
# === Component 7: Snapshot Publishing ===

def build_snapshot(positions: list[PositionInfo], account_id="12345",
                   broker="TestBroker", server="TestServer",
                   balance=10000, equity=10050) -> dict:
    """Mirror of BuildFullSnapshotJson()."""
    snapshot = {
        "type": "SNAPSHOT",
        "timestamp": datetime.now().strftime("%Y.%m.%d %H:%M:%S"),
        "platform": "MT5",
        "role": "master",
        "accountId": account_id,
        "broker": broker,
        "server": server,
        "balance": balance,
        "equity": equity,
        "margin": 100.0,
        "freeMargin": equity - 100.0,
        "marginLevel": (equity / 100.0) * 100 if 100.0 > 0 else None,
        "floatingPnL": equity - balance,
        "currency": "USD",
        "leverage": 500,
        "status": "Licensed - Master Active",
        "isLicenseValid": True,
        "isPaused": False,
        "zmqMode": True,
        "eventDriven": True,
        "positions": [build_position_json(p) for p in positions],
    }
    return snapshot

def build_position_json(p: PositionInfo) -> dict:
    """Mirror of BuildPositionsJson() per-position block."""
    return {
        "id": str(p.ticket),
        "symbol": p.symbol,
        "volume": int(p.volume * 100000),
        "volumeLots": p.volume,
        "side": p.pos_type,
        "entryPrice": p.entry_price,
        "currentPrice": p.current_price,
        "stopLoss": p.stop_loss if p.stop_loss > 0 else None,
        "takeProfit": p.take_profit if p.take_profit > 0 else None,
        "profit": p.profit,
        "swap": p.swap,
        "commission": p.commission,
        "openTime": p.open_time,
        "comment": p.comment,
    }

# --- Test ---
test_positions = [
    PositionInfo(1001, "EURUSD", 0.10, "BUY", 1.0950, 1.0960, 1.0900, 1.1000, 10.0, 0, 0, "2025.01.01 10:00:00", ""),
    PositionInfo(1002, "GBPUSD", 0.05, "SELL", 1.2700, 1.2690, 1.2750, 1.2650, 5.0, -0.50, 0, "2025.01.01 11:00:00", ""),
]

snapshot = build_snapshot(test_positions)
print("=== Snapshot (pretty-printed) ===")
print(json_mod.dumps(snapshot, indent=2)[:1500])
print(f"\nPositions in snapshot: {len(snapshot['positions'])}")
print(f"Wire size: {len(json_mod.dumps(snapshot))} bytes")

---
## Component 8 — OnTradeTransaction (Instant Event Detection)

This is the heart of the Master EA. MT5 calls `OnTradeTransaction()` for every
deal/order/position change. The EA filters for `TRADE_TRANSACTION_DEAL_ADD` to
publish instant open/close/reverse events.

**MQ5 Function:** `OnTradeTransaction()`

Deal entry types:
- `DEAL_ENTRY_IN` → `POSITION_OPENED`
- `DEAL_ENTRY_OUT` → `POSITION_CLOSED`
- `DEAL_ENTRY_INOUT` → `POSITION_REVERSED`

In [ ]:
# === Component 8: Trade Transaction Simulation ===

def simulate_trade_transaction(deal_id: int, entry: str, deal_type: str,
                                symbol: str, volume: float, price: float,
                                profit: float, position_id: int,
                                sl: float = 0, tp: float = 0,
                                publisher: EventPublisher = None) -> str | None:
    """
    Simulate MT5's OnTradeTransaction for a DEAL_ADD event.
    Returns the event type published, or None.
    """
    # Skip non-trade deals
    if deal_type not in ("BUY", "SELL"):
        return None
    
    data = {
        "deal": deal_id,
        "position": position_id,
        "symbol": symbol,
        "volume": volume,
        "price": price,
        "profit": profit,
        "type": deal_type,
        "stopLoss": sl if sl > 0 else None,
        "takeProfit": tp if tp > 0 else None,
        "entry": entry,
    }
    
    event_map = {
        "IN": "POSITION_OPENED",
        "OUT": "POSITION_CLOSED",
        "INOUT": "POSITION_REVERSED",
    }
    
    event_type = event_map.get(entry)
    if not event_type:
        return None
    
    print(f">> {event_type}: {symbol} {deal_type} {volume:.2f} @ {price:.5f}", end="")
    if entry == "OUT":
        print(f" P&L={profit:.2f}", end="")
    print(f" #{position_id}")
    
    if publisher:
        publisher.publish_event(event_type, data)
    
    return event_type

# --- Test: Simulate a trade lifecycle ---
pub2 = EventPublisher("12345")

print("=== Trade Lifecycle Simulation ===")
print("")

# 1. Open BUY EURUSD
simulate_trade_transaction(
    deal_id=5001, entry="IN", deal_type="BUY",
    symbol="EURUSD", volume=0.10, price=1.09500, profit=0,
    position_id=4001, sl=1.09000, tp=1.10000, publisher=pub2
)

# 2. Open SELL GBPUSD
simulate_trade_transaction(
    deal_id=5002, entry="IN", deal_type="SELL",
    symbol="GBPUSD", volume=0.05, price=1.27000, profit=0,
    position_id=4002, sl=1.27500, tp=1.26500, publisher=pub2
)

# 3. Close EURUSD with profit
simulate_trade_transaction(
    deal_id=5003, entry="OUT", deal_type="SELL",
    symbol="EURUSD", volume=0.10, price=1.09800, profit=30.0,
    position_id=4001, publisher=pub2
)

print(f"\nTotal events published: {pub2.event_index}")
for et, msg in pub2.published:
    print(f"  [{msg['eventIndex']}] {et} - {msg['data'].get('symbol', '')}")

---
## Component 9 — Command Processing

The Electron app sends commands via ZMQ REQ/REP to the Master's port 51811.

| Action | Description |
|--------|-------------|
| `PAUSE` | Pause event publishing |
| `RESUME` | Resume publishing |
| `STATUS` | Full snapshot response |
| `PING` | Keepalive |
| `CONFIG` | Get ZMQ/publish config |
| `GET_HISTORY` | Trade deal history |
| `GET_CURVE_KEY` | CURVE public key (if enabled) |

**MQ5 Function:** `ProcessCommands()`

In [ ]:
# === Component 9: Command Processing ===

def send_master_command(port: int, command: dict, timeout_ms: int = 3000) -> dict | None:
    """Send a JSON command to the master's REP socket."""
    ctx = zmq.Context()
    req = ctx.socket(zmq.REQ)
    req.setsockopt(zmq.LINGER, 100)
    req.setsockopt(zmq.RCVTIMEO, timeout_ms)
    req.setsockopt(zmq.SNDTIMEO, timeout_ms)
    endpoint = f"tcp://localhost:{port}"
    req.connect(endpoint)
    try:
        req.send_string(json_mod.dumps(command))
        reply = req.recv_string()
        return json_mod.loads(reply)
    except zmq.error.Again:
        print(f"Timeout from {endpoint}")
        return None
    finally:
        req.close()
        ctx.term()

# --- Test commands (run when MT5 EA is active) ---
cmd_port = master_config["InpCommandPort"]
print(f"Master command port: {cmd_port}")
print("\nTest commands (uncomment to run against live EA):")
print('  send_master_command(cmd_port, {"action": "PING"})')
print('  send_master_command(cmd_port, {"action": "STATUS"})')
print('  send_master_command(cmd_port, {"action": "CONFIG"})')
print('  send_master_command(cmd_port, {"action": "GET_HISTORY", "days": 30})')

# Uncomment to test:
# result = send_master_command(cmd_port, {"action": "PING"})
# print(f"PING: {result}")

---
## Component 10 — Account Data & History

The Master builds rich account data JSON for `CONNECTED` and `ACCOUNT_UPDATE` events,
and can serve historical deal data via `GET_HISTORY`.

**MQ5 Functions:** `BuildAccountDataJson()`, `BuildHistoryResponse()`

In [ ]:
# === Component 10: Account Data ===

def build_account_data(positions: list[PositionInfo], account_id="12345",
                       broker="TestBroker", server="TestServer",
                       balance=10000, equity=10050) -> dict:
    """Mirror of BuildAccountDataJson()."""
    return {
        "accountId": account_id,
        "broker": broker,
        "server": server,
        "balance": balance,
        "equity": equity,
        "margin": 100.0,
        "freeMargin": equity - 100.0,
        "marginLevel": round(equity / 100.0 * 100, 2),
        "floatingPnL": equity - balance,
        "currency": "USD",
        "leverage": 500,
        "status": "Licensed - Master Active",
        "isLicenseValid": True,
        "isPaused": False,
        "eventDriven": True,
        "positions": [build_position_json(p) for p in positions],
    }

def build_history_response(deals: list[dict], account_id="12345") -> dict:
    """Mirror of BuildHistoryResponse() — formats deal history."""
    return {
        "success": True,
        "action": "GET_HISTORY",
        "accountId": account_id,
        "deals": deals,
        "timestamp": datetime.now().strftime("%Y.%m.%d %H:%M:%S"),
    }

# --- Test ---
acct_data = build_account_data(test_positions)
print("=== Account Data ===")
print(json_mod.dumps(acct_data, indent=2)[:800])

test_deals = [
    {"ticket": 5001, "positionId": 4001, "symbol": "EURUSD", "type": "BUY",
     "entry": "IN", "volume": 0.10, "price": 1.09500, "profit": 0, "swap": 0, "commission": -0.70,
     "time": "2025.01.01 10:00:00", "comment": ""},
    {"ticket": 5002, "positionId": 4001, "symbol": "EURUSD", "type": "SELL",
     "entry": "OUT", "volume": 0.10, "price": 1.09800, "profit": 30.0, "swap": 0, "commission": -0.70,
     "time": "2025.01.01 12:00:00", "comment": ""},
]

history = build_history_response(test_deals)
print(f"\n=== History Response ({len(history['deals'])} deals) ===")
for d in history["deals"]:
    print(f"  #{d['ticket']} {d['entry']} {d['type']} {d['symbol']} {d['volume']} @ {d['price']} P&L={d['profit']}")

---
## Component 11 — Registration File (Electron App Discovery)

The Master writes `HedgeEdge/<login>.json` to Common Files so the Electron
app can discover it and connect.

**MQ5 Function:** `WriteRegistrationFile()`

In [ ]:
# === Component 11: Registration File ===

def build_master_registration(login: str, broker: str, server: str, config: dict,
                               curve_enabled: bool = False, curve_key: str = "") -> dict:
    """Mirror of MQ5 WriteRegistrationFile() for master."""
    reg = {
        "login": login,
        "broker": broker,
        "server": server,
        "dataPort": config["InpDataPort"],
        "commandPort": config["InpCommandPort"],
        "role": "master",
        "version": "3.0",
        "eventDriven": True,
        "curveEnabled": curve_enabled,
        "timestamp": datetime.now().strftime("%Y.%m.%d %H:%M:%S"),
    }
    if curve_enabled and curve_key:
        reg["curvePublicKey"] = curve_key
    return reg

def check_registration_files():
    common = find_mt5_common_files()
    if not common:
        print("Common Files not found")
        return []
    he_dir = os.path.join(common, "HedgeEdge")
    if not os.path.isdir(he_dir):
        print("No HedgeEdge directory")
        return []
    files = [f for f in os.listdir(he_dir) if f.endswith(".json")]
    print(f"Found {len(files)} registration file(s):")
    for f in files:
        with open(os.path.join(he_dir, f), "r") as fh:
            data = json_mod.load(fh)
        print(f"  {f}: role={data.get('role')} login={data.get('login')}")
    return files

# --- Test ---
reg = build_master_registration("12345", "TestBroker", "TestServer", master_config)
print("Master Registration JSON:")
print(json_mod.dumps(reg, indent=2))

print("\n--- Existing registrations ---")
check_registration_files()

---
## Component 12 — End-to-End Publisher Test

Spins up a mock Master publisher that sends events, heartbeats, and snapshots.
Use this to test the Slave EA or the Electron app connector without a real MT5 terminal.

In [ ]:
# === Component 12: Mock Master Publisher ===
import time

def run_mock_master(data_port: int, duration_seconds: int = 30,
                    heartbeat_interval: int = 5):
    """
    Run a mock master that publishes heartbeats and snapshots.
    Useful for testing the slave EA or Electron app without a real MT5.
    """
    print(f"=== Mock Master Publisher ===")
    print(f"  PUB port: {data_port}")
    print(f"  Duration: {duration_seconds}s")
    
    ctx = zmq.Context()
    pub = ctx.socket(zmq.PUB)
    pub.setsockopt(zmq.LINGER, 100)
    pub.setsockopt(zmq.SNDHWM, 1000)
    pub.bind(f"tcp://*:{data_port}")
    
    publisher = EventPublisher("MOCK-12345")
    mock_positions = [
        PositionInfo(9001, "EURUSD", 0.10, "BUY", 1.0950, 1.0960, 1.0900, 1.1000, 10.0, 0, 0, "2025.01.01 10:00:00", ""),
    ]
    
    # Publish CONNECTED event
    connected_data = build_account_data(mock_positions, account_id="MOCK-12345")
    msg = publisher.publish_event("CONNECTED", connected_data)
    pub.send_string(msg)
    print(f"  Published CONNECTED")
    
    start = time.time()
    hb_count = 0
    snap_count = 0
    
    while time.time() - start < duration_seconds:
        # Heartbeat
        hb_msg = publisher.publish_heartbeat()
        pub.send_string(hb_msg)
        hb_count += 1
        
        # Snapshot
        snapshot = build_snapshot(mock_positions, account_id="MOCK-12345")
        snap_msg = f"SNAPSHOT|{json_mod.dumps(snapshot)}"
        pub.send_string(snap_msg)
        snap_count += 1
        
        print(f"  [{hb_count}] Heartbeat + Snapshot", end="\r")
        time.sleep(heartbeat_interval)
    
    # Disconnect
    dc_msg = publisher.publish_event("DISCONNECTED", {"reason": 0})
    pub.send_string(dc_msg)
    
    pub.close()
    ctx.term()
    print(f"\n  Done. Sent {hb_count} heartbeats, {snap_count} snapshots")

# --- Run mock master (uncomment when needed) ---
# run_mock_master(data_port=51810, duration_seconds=30, heartbeat_interval=5)

---
## Component 13 — Live Connection Test

Test against a real HE_Prop EA running on MT5.

In [ ]:
# === Component 13: Live Connection to HE_Prop ===

def test_master_connection(cmd_port: int, data_port: int, listen_seconds: int = 10):
    """Test connectivity to a running HE_Prop EA."""
    print("=== Live Master Connection Test ===")
    
    # 1. PING
    print("\n[1] PING...")
    ping = send_master_command(cmd_port, {"action": "PING"})
    if ping and ping.get("pong"):
        print(f"  Master responded: role={ping.get('role')}")
    else:
        print("  Master not responding (is HE_Prop.mq5 running on MT5?)")
        return
    
    # 2. CONFIG
    print("\n[2] CONFIG...")
    config = send_master_command(cmd_port, {"action": "CONFIG"})
    if config:
        c = config.get("config", {})
        print(f"  Data port: {c.get('dataPort')}")
        print(f"  CURVE: {c.get('curveEnabled')}")
        print(f"  Event driven: {c.get('eventDriven')}")
    
    # 3. STATUS (full snapshot)
    print("\n[3] STATUS...")
    status = send_master_command(cmd_port, {"action": "STATUS"})
    if status:
        print(f"  Account: {status.get('accountId')}")
        print(f"  Broker: {status.get('broker')}")
        print(f"  Balance: {status.get('balance')}")
        print(f"  Equity: {status.get('equity')}")
        positions = status.get("positions", [])
        print(f"  Positions: {len(positions)}")
        for p in positions[:5]:
            print(f"    #{p['id']} {p['symbol']} {p['side']} {p.get('volumeLots', '?')} lots P&L={p.get('profit', 0)}")
    
    # 4. Subscribe to PUB for events
    print(f"\n[4] Listening on PUB port {data_port} for {listen_seconds}s...")
    ctx = zmq.Context()
    sub = ctx.socket(zmq.SUB)
    sub.setsockopt(zmq.LINGER, 100)
    sub.setsockopt(zmq.RCVTIMEO, 1)
    sub.setsockopt_string(zmq.SUBSCRIBE, "EVENT|")
    sub.setsockopt_string(zmq.SUBSCRIBE, "SNAPSHOT|")
    sub.connect(f"tcp://localhost:{data_port}")
    
    events = {"EVENT": 0, "SNAPSHOT": 0}
    end_time = time.time() + listen_seconds
    while time.time() < end_time:
        try:
            raw = sub.recv_string(flags=zmq.NOBLOCK)
            topic = raw.split("|", 1)[0]
            events[topic] = events.get(topic, 0) + 1
        except zmq.error.Again:
            time.sleep(0.05)
    
    sub.close()
    ctx.term()
    print(f"  Received: {events}")

# --- Run (uncomment when HE_Prop is running on MT5) ---
# test_master_connection(
#     cmd_port=master_config["InpCommandPort"],
#     data_port=master_config["InpDataPort"],
#     listen_seconds=15
# )

---
## Dashboard (Component 14) — Visual Panel

The MQ5 renders a branded overlay showing:
- **Status**: Connected / Paused / License Error
- **Positions**: Open position count
- **Published**: Total publish count

This is purely MQL5 chart rendering — no Python equivalent needed.
The Electron app gets all data via the command/event channels above.

---
## Deployment Checklist

Before deploying HE_Prop.mq5 to a live MT5 terminal:

1. **DLL files in place:**
   - `MQL5/Libraries/libzmq.dll`
   - `MQL5/Libraries/libsodium.dll`
   - `MQL5/Libraries/HedgeEdgeLicense.dll` (optional)

2. **Include files:**
   - `MQL5/Include/ZMQv2.mqh`

3. **MT5 Settings:**
   - Tools > Options > Expert Advisors > Allow DLL imports
   - Tools > Options > Expert Advisors > Allow WebRequest for: `https://hedge-edge-app-backend-production.up.railway.app`

4. **License:**
   - Place key in EA input OR in `Common Files/HedgeEdge/license.key`
   - OR enable DEV MODE for testing

5. **Compile:** Use MQL Tools extension in VS Code or MT5 MetaEditor

6. **Attach:** Drag compiled `.ex5` onto a chart — it will auto-bind PUB on port 51810

7. **Verify:** Run Component 13 above to confirm connectivity